# Exploring cratebank with DuckDB

This notebook queries cratebank's public Parquet files directly—no account or local dataset download is required. It starts with dataset health, then looks at Cargo unit wall time, samply compiler-phase samples, build settings, and one build timeline.

Cargo durations are wall-clock measurements per unit. Samply phase values are CPU-weighted sample counts. Keep those quantities separate.

In [ ]:
%pip install -q duckdb pandas matplotlib

In [ ]:
from urllib.request import urlopen

import duckdb
import matplotlib.pyplot as plt
from IPython.display import display

INSTALL_SQL = "https://raw.githubusercontent.com/PowderworksCode/cratebank/main/docs/install.sql"
con = duckdb.connect()
con.execute(urlopen(INSTALL_SQL).read().decode("utf-8"))
print("cratebank views installed")

## Dataset health

Start by checking how many builds and units are available, the observation window, and how many units were withheld by the privacy filter.

In [ ]:
health = con.sql("""
    SELECT
        count(*) AS builds,
        min(try_cast(timestamp AS TIMESTAMPTZ)) AS first_build,
        max(try_cast(timestamp AS TIMESTAMPTZ)) AS latest_build,
        sum(units) AS public_units,
        sum(units_withheld) AS withheld_units,
        count(DISTINCT rustc_version) AS compiler_versions,
        count(DISTINCT machine_id) AS identified_machines
    FROM sessions
""").df()
display(health)

## Package wall time

Summing unit durations measures accumulated unit wall-seconds, not end-to-end build time: units can overlap. The median is useful alongside the total so frequently observed packages do not dominate silently.

In [ ]:
package_wall = con.sql("""
    SELECT
        package,
        count(*) AS observations,
        round(sum(duration), 2) AS total_unit_wall_s,
        round(median(duration), 3) AS median_unit_wall_s,
        round(max(duration), 3) AS max_unit_wall_s
    FROM units
    WHERE duration IS NOT NULL
    GROUP BY package
    ORDER BY total_unit_wall_s DESC
    LIMIT 20
""").df()
display(package_wall)

ax = package_wall.sort_values("total_unit_wall_s").plot.barh(
    x="package", y="total_unit_wall_s", legend=False, figsize=(9, 7)
)
ax.set(title="Accumulated Cargo unit wall time", xlabel="wall-seconds", ylabel="")
plt.tight_layout()

## Where rustc spends CPU samples

This view uses serial rustc threads only. Parallel codegen threads are intentionally separate because combining them would mix distinct thread classes.

In [ ]:
phase_share = con.sql("""
    SELECT
        phase,
        sum(samples) AS samples,
        round(100.0 * sum(samples) / sum(sum(samples)) OVER (), 1) AS share_pct
    FROM phases
    WHERE thread = 'serial'
    GROUP BY phase
    ORDER BY samples DESC
""").df()
display(phase_share)

ax = phase_share.sort_values("share_pct").plot.barh(
    x="phase", y="share_pct", legend=False, figsize=(8, 5)
)
ax.set(title="Serial rustc CPU sample share", xlabel="percent of samples", ylabel="")
plt.tight_layout()

## Build settings in the wild

These rows come from scrubbed rustc command lines. Path values and full command lines are never published.

In [ ]:
settings = con.sql("""
    SELECT flag, value, count(*) AS units
    FROM unit_flags
    GROUP BY flag, value
    ORDER BY units DESC, flag, value
    LIMIT 30
""").df()
display(settings)

## Inspect one build timeline

Cargo records concurrency and whole-machine CPU on different clocks. They are plotted on separate axes without pretending that samples at the same row index happened together.

In [ ]:
candidate = con.sql("""
    SELECT run_id
    FROM timeline
    GROUP BY run_id
    ORDER BY count(*) DESC
    LIMIT 1
""").fetchone()

if candidate is None:
    print("No timeline rows are available yet.")
else:
    run_id = candidate[0]
    concurrency = con.execute("""
        SELECT t, active, waiting, inactive
        FROM timeline
        WHERE run_id = ? AND active IS NOT NULL
        ORDER BY t
    """, [run_id]).df()
    cpu = con.execute("""
        SELECT t, cpu_pct
        FROM timeline
        WHERE run_id = ? AND cpu_pct IS NOT NULL
        ORDER BY t
    """, [run_id]).df()

    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    if not concurrency.empty:
        concurrency.plot(x="t", y=["active", "waiting", "inactive"], ax=axes[0])
    axes[0].set(title=f"Cargo concurrency — {run_id}", ylabel="units")
    if not cpu.empty:
        cpu.plot(x="t", y="cpu_pct", ax=axes[1], legend=False)
    axes[1].set(title="Whole-machine CPU", xlabel="seconds since build start", ylabel="percent")
    plt.tight_layout()

## Keep exploring

All five views join through `run_id`. Useful next cuts include compiler version, profile, target, feature set, machine class, and CI versus local builds. Treat the anonymous contributions as observational data rather than a representative sample of all Rust builds.